# FireSight-IR | Module 2 — LinkedIn Visualization

Run this notebook after Module 2 completes to generate a publication-quality figure for LinkedIn.

**Depends on:** Module 2 outputs in Google Drive
- `data/scalers/class_weights.json` or `class_weights_v2.json`
- `data/processed/viirs_fp_v2/` parquets
- `data/scalers/feature_scalers.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install pandas numpy matplotlib scipy pyarrow -q

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from scipy import stats
warnings.filterwarnings('ignore')
print('Imports OK.')

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
V2_DIR      = f'{BASE_DIR}/data/processed/viirs_fp_v2'
FIGURES_DIR = f'{BASE_DIR}/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load class weights (v2 preferred, fall back to v1)
wpath = f'{BASE_DIR}/data/scalers/class_weights_v2.json'
if not os.path.exists(wpath):
    wpath = f'{BASE_DIR}/data/scalers/class_weights.json'
with open(wpath) as f:
    cw = json.load(f)
if isinstance(cw, list):
    weights = cw
elif isinstance(cw, dict):
    weights = cw.get('_list', [float(cw.get(str(i), [6.9,0.36,9.81][i])) for i in range(3)])
else:
    weights = [18.98, 1.0, 27.02]

ALL_YEARS = [2018, 2019, 2020, 2021, 2022, 2023]
TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]

# Load all parquets
dfs = []
for year in ALL_YEARS:
    fp = f'{V2_DIR}/viirs_fp_v2_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['year'] = year
        dfs.append(df)
        print(f'  {year}: {len(df):,} pixels')

df_all = pd.concat(dfs, ignore_index=True)
print(f'Total: {len(df_all):,} pixels')

# Compute BTD if not present
if 'BTD' not in df_all.columns and 'BT_I4' in df_all.columns and 'BT_I5' in df_all.columns:
    df_all['BTD'] = df_all['BT_I4'] - df_all['BT_I5']

print('Data loaded.')

---
## LinkedIn hero figure — Module 2

In [ ]:
# ── Design tokens ─────────────────────────────────────────────────────────────
BG      = '#0D1117'
PANEL   = '#161B22'
BORDER  = '#30363D'
TEXT    = '#E6EDF3'
SUBTEXT = '#8B949E'
ORANGE  = '#E85D04'
NAVY    = '#1565C0'
TEAL    = '#0D9488'
GREEN   = '#16A34A'
RED     = '#DC2626'
AMBER   = '#D97706'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': BORDER, 'text.color': TEXT,
    'xtick.color': SUBTEXT, 'ytick.color': SUBTEXT,
    'axes.labelcolor': SUBTEXT, 'grid.color': BORDER, 'grid.linewidth': 0.5,
})

# ── Data prep ──────────────────────────────────────────────────────────────────
# Label counts by year
year_label = df_all.groupby(['year', 'label']).size().unstack(fill_value=0)
years_plot = sorted(df_all['year'].unique())

wf_yr = [int(year_label.get(1, {}).get(y, 0)) for y in years_plot]
fa_yr = [int(year_label.get(2, {}).get(y, 0)) for y in years_plot]
nf_yr = [int(year_label.get(0, {}).get(y, 0)) for y in years_plot]

# Overall label distribution
lbl_counts = df_all['label'].value_counts().sort_index()
total_px   = len(df_all)

# Feature groups for coverage plot
feat_groups = [
    ('ERA5 atmospheric (16)',  GREEN,  True,  '100% — T, q profiles, BLH, TCWV'),
    ('MODIS land cover (8)',   GREEN,  True,  '100% — MCD12Q1 v6.1 via earthaccess'),
    ('OSM infrastructure (5)', GREEN,  True,  '87% — industrial/urban proximity'),
    ('Solar geometry (2)',     GREEN,  True,  '100% — sol_zen, is_day'),
    ('Derived physics (6)',    GREEN,  True,  '100% — AOD, LI, DOY, BT anomaly'),
    ('DNB emitters (2)',       RED,    False, '0% — VNP46A2 parse failed'),
    ('Burn scars (2)',         RED,    False, '0% — MTBS API returned empty'),
    ('FIRMS type (1)',         RED,    False, '0% — VNP14IMG default = 0'),
]
n_real = sum(1 for g in feat_groups if g[2])
n_zero = sum(1 for g in feat_groups if not g[2])
n_feat_total = 42  # from module 2

# ── Build figure: 2 rows × 3 cols ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor=BG)
fig.patch.set_facecolor(BG)

gs = gridspec.GridSpec(2, 3, figure=fig,
                       left=0.06, right=0.97,
                       top=0.86, bottom=0.09,
                       wspace=0.35, hspace=0.52)

# ── Panel A: Annual fire pixels stacked bar ────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.set_facecolor(PANEL)
x = np.arange(len(years_plot))
ax_a.bar(x, wf_yr, color=ORANGE, alpha=0.88, label='Wildfire',     width=0.65)
ax_a.bar(x, fa_yr, color=NAVY,   alpha=0.88, label='False-alarm',  width=0.65, bottom=wf_yr)
ax_a.bar(x, nf_yr, color=TEAL,   alpha=0.88, label='Non-fire',     width=0.65,
         bottom=[w+f for w,f in zip(wf_yr, fa_yr)])
ax_a.set_xticks(x); ax_a.set_xticklabels(years_plot, fontsize=9)
ax_a.yaxis.set_major_formatter(FuncFormatter(lambda v,_: f'{v/1e3:.0f}K'))
ax_a.set_title('Fire pixels by year & label', color=TEXT, fontsize=11, pad=8)
ax_a.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8, loc='upper right')
ax_a.spines[['top','right']].set_visible(False)
ax_a.grid(axis='y', alpha=0.25)
# Annotate 2020 peak
peak_idx = wf_yr.index(max(wf_yr))
ax_a.annotate('2020\nmega-fire', xy=(peak_idx, wf_yr[peak_idx]),
              xytext=(peak_idx+0.6, wf_yr[peak_idx]*0.85),
              fontsize=7.5, color=SUBTEXT,
              arrowprops=dict(arrowstyle='->', color=SUBTEXT, lw=1))
ax_a.text(len(years_plot)-1, -max(wf_yr)*0.12, '← 2023: held-out',
          ha='right', fontsize=7.5, color=SUBTEXT, style='italic')

# ── Panel B: Label donut chart ─────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
ax_b.set_facecolor(PANEL)
lbl_vals  = [lbl_counts.get(i, 0) for i in range(3)]
lbl_names = ['Non-fire\n(4.82%)', 'Wildfire\n(91.52%)', 'False-alarm\n(3.65%)']
lbl_cols  = [TEAL, ORANGE, NAVY]
wedges, texts = ax_b.pie(lbl_vals, colors=lbl_cols, startangle=90,
                          wedgeprops=dict(width=0.55, edgecolor=BG, linewidth=2),
                          counterclock=False)
ax_b.set_title('Label distribution\n(1,149,722 total pixels)', color=TEXT, fontsize=11, pad=8)
# Custom legend
leg_handles = [mpatches.Patch(color=c, label=n) for c,n in zip(lbl_cols, lbl_names)]
ax_b.legend(handles=leg_handles, facecolor=PANEL, edgecolor=BORDER,
            labelcolor=TEXT, fontsize=8, loc='lower center',
            bbox_to_anchor=(0.5, -0.22), ncol=1)
# Centre text
ax_b.text(0, 0, '42,018\nfalse-alarms\nidentified', ha='center', va='center',
          fontsize=8.5, color=NAVY, fontweight='bold')

# ── Panel C: Class weights bar ─────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
ax_c.set_facecolor(PANEL)
cls_names = ['Non-fire', 'Wildfire', 'False-alarm']
cls_cols  = [TEAL, ORANGE, NAVY]
bars = ax_c.bar(cls_names, weights, color=cls_cols, alpha=0.88, width=0.55)
for bar, w in zip(bars, weights):
    ax_c.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
              f'×{w:.1f}', ha='center', fontsize=11, color=TEXT, fontweight='bold')
ax_c.set_title('Loss weights (CE)\nHigher = penalised more for missing', color=TEXT, fontsize=11, pad=8)
ax_c.set_ylabel('Weight multiplier', color=SUBTEXT, fontsize=9)
ax_c.spines[['top','right']].set_visible(False)
ax_c.grid(axis='y', alpha=0.25)
ax_c.text(0.5, 0.97,
          'False-alarm is the rarest class\nand carries the highest penalty',
          transform=ax_c.transAxes, ha='center', va='top',
          fontsize=8, color=SUBTEXT, style='italic')

# ── Panel D: Feature coverage horizontal bar ───────────────────────────────────
ax_d = fig.add_subplot(gs[1, :])
ax_d.set_facecolor(PANEL)

feat_labels = [g[0] for g in feat_groups]
feat_colors = [g[1] for g in feat_groups]
feat_real   = [g[2] for g in feat_groups]
feat_desc   = [g[3] for g in feat_groups]

y_pos = np.arange(len(feat_groups))
bar_height = 0.62

for i, (label, color, is_real, desc) in enumerate(feat_groups):
    alpha = 0.85 if is_real else 0.6
    ax_d.barh(i, 1, height=bar_height, color=color, alpha=alpha,
              edgecolor=BORDER, linewidth=0.5)
    status = '✓  Real data' if is_real else '○  Zero-filled'
    ax_d.text(0.012, i, f'{label}', va='center', ha='left',
              fontsize=10, color='white', fontweight='bold')
    ax_d.text(0.988, i, desc, va='center', ha='right',
              fontsize=8.5, color='white' if is_real else '#FFAAAA')

ax_d.set_xlim(0, 1.0)
ax_d.set_yticks(y_pos)
ax_d.set_yticklabels([])
ax_d.set_xticks([])
ax_d.invert_yaxis()
ax_d.spines[['top','right','bottom','left']].set_visible(False)
ax_d.set_title(f'Feature coverage — {n_real} real  ·  {n_zero} zero-filled  ·  {n_feat_total} total features',
               color=TEXT, fontsize=11, pad=8, loc='left')

# Legend
ax_d.legend(handles=[
    mpatches.Patch(color=GREEN, label='Real sensor data'),
    mpatches.Patch(color=RED, label='Zero-filled (API failure)')],
    facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9,
    loc='lower right')

# ── Header ─────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.97,
         'FireSight-IR  |  Module 2 — Feature Engineering',
         ha='center', va='top', fontsize=16, fontweight='bold', color=TEXT)
fig.text(0.5, 0.93,
         '1,149,722 VIIRS fire pixels  ·  42 features  ·  Robust normalisation  ·  Physics-informed class weights  ·  Western CONUS 2018–2023',
         ha='center', va='top', fontsize=10, color=SUBTEXT)

# ── Footer ─────────────────────────────────────────────────────────────────────
fig.text(0.5, 0.015,
         'FireSight-IR  ·  github.com/Ibekwemmanuel7  ·  FireSat Protoflight-aligned wildfire detection  ·  VIIRS + ERA5 + MODIS',
         ha='center', va='bottom', fontsize=8, color=SUBTEXT)

out = f'{FIGURES_DIR}/02_linkedin_hero.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'\n✓ LinkedIn figure saved → {out}')
print(f'  Recommended: use this image for both LinkedIn and WhatsApp')

---
## BTD separation detail figure (optional additional post)

In [ ]:
if 'BTD' in df_all.columns and not df_all['BTD'].isna().all():
    fig2, ax = plt.subplots(figsize=(13, 6), facecolor=BG)
    ax.set_facecolor(PANEL)

    for lbl_val, name, color, lw in [
        (1, f'Wildfire  (n={lbl_counts.get(1,0):,})', ORANGE, 2.5),
        (2, f'False-alarm  (n={lbl_counts.get(2,0):,})', NAVY, 2.5),
        (0, f'Non-fire  (n={lbl_counts.get(0,0):,})', TEAL, 1.5),
    ]:
        sub = df_all[df_all['label'] == lbl_val]['BTD'].dropna()
        if len(sub) == 0: continue
        s = sub.sample(min(50000, len(sub)), random_state=42)
        x_r = np.linspace(s.quantile(0.005), s.quantile(0.995), 400)
        kde  = stats.gaussian_kde(s, bw_method=0.25)
        dens = kde(x_r)
        dens = dens / dens.max()
        ax.fill_between(x_r, dens, alpha=0.25, color=color)
        ax.plot(x_r, dens, color=color, linewidth=lw, label=name)
        ax.axvline(s.median(), color=color, linewidth=1, linestyle='--', alpha=0.6)
        ax.text(s.median()+0.3, 0.97, f'med={s.median():.0f}K',
                va='top', fontsize=8, color=color, alpha=0.8)

    ax.axvline(10, color='white', linewidth=1.8, linestyle=':', alpha=0.7, label='BTD=10K threshold')
    ax.axvline(20, color='#888888', linewidth=1, linestyle=':', alpha=0.5, label='BTD=20K label gate')
    ax.set_xlabel('BTD  =  BT_I4 − BT_I5  (Kelvin)', fontsize=11, color=SUBTEXT)
    ax.set_ylabel('Normalised density', fontsize=11, color=SUBTEXT)
    ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(alpha=0.2)

    fig2.text(0.5, 0.99,
              'FireSight-IR  |  Brightness Temperature Difference separation',
              ha='center', va='top', fontsize=14, color=TEXT, fontweight='bold')
    fig2.text(0.5, 0.95,
              'Wildfire vs false-alarm vs non-fire  ·  20+ K median separation  ·  Physics threshold at BTD=10K',
              ha='center', va='top', fontsize=10, color=SUBTEXT)

    out2 = f'{FIGURES_DIR}/02_btd_separation.png'
    plt.savefig(out2, dpi=200, bbox_inches='tight', facecolor=BG)
    plt.show()
    print(f'✓ BTD separation figure saved → {out2}')
else:
    print('BTD column not available in parquets — skipping BTD figure.')
    print('The hero figure (02_linkedin_hero.png) is sufficient for posting.')